# RoBERTa fine-tuning for phishing detection
**How to run:** 1) Runtime -> Change runtime type -> **T4 GPU** -> Save.  2) Runtime -> **Run all**.
Wait about 15 to 25 minutes. At the end a results table appears and `roberta_results.csv` downloads.

In [ ]:
!pip install -q transformers scikit-learn pandas
!rm -rf /content/Phishing && git clone -q https://github.com/krishitha11/Phishing.git /content/Phishing
!ls /content/Phishing/data

In [ ]:
DATA_DIR = '/content/Phishing/data'
MODEL_NAME = 'roberta-base'
MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 2
LR = 2e-5
MAX_TRAIN_ROWS = None   # data folder already holds 20,000 training emails
SEED = 42

In [ ]:
import torch, random, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)
assert device == 'cuda', 'No GPU! Runtime -> Change runtime type -> T4 GPU, then run again from Cell 1.'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
val = pd.read_csv(f'{DATA_DIR}/val.csv').sample(3000, random_state=SEED)  # smaller, to save time
if MAX_TRAIN_ROWS:
    train = train.sample(MAX_TRAIN_ROWS, random_state=SEED)
train['text'] = train['text'].astype(str); val['text'] = val['text'].astype(str)
print(len(train), 'training emails,', len(val), 'validation emails')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

In [ ]:
def predict(texts, batch_size=128):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(list(texts[i:i+batch_size]), truncation=True, max_length=MAX_LEN,
                      padding=True, return_tensors='pt').to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**enc).logits
            preds.extend(logits.argmax(-1).cpu().tolist())
    return np.array(preds)

In [ ]:
texts = train['text'].tolist(); labels = train['label'].tolist()
steps = (len(texts) // BATCH_SIZE) * EPOCHS
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    order = np.random.permutation(len(texts))
    running = 0.0
    for step, i in enumerate(range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE)):
        idx = order[i:i + BATCH_SIZE]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(device)
        y = torch.tensor([labels[j] for j in idx]).to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            loss = model(**enc, labels=y).loss
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()
        if (step + 1) % 100 == 0:
            print(f'epoch {epoch+1}  step {step+1}  loss {running/100:.4f}')
            running = 0.0
    val_acc = (predict(val['text'].tolist()) == val['label'].values).mean()
    print(f'=== epoch {epoch+1} done. Validation accuracy: {val_acc:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

def score(name, df):
    pred = predict(df['text'].astype(str).tolist())
    y = df['label'].values
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(test_set=name, accuracy=accuracy_score(y, pred), precision=p, recall=r, f1=f1,
                false_positive_rate=fp / (fp + tn) if (fp + tn) else 0.0, rows=len(df))

files = {'clean': 'test.csv', 'typos': 'test_typos.csv', 'synonyms': 'test_synonyms.csv',
         'char_swaps': 'test_char_swaps.csv', 'all_combined': 'test_all_combined.csv',
         'external': 'test_external.csv'}
rows = []
for name, f in files.items():
    rows.append(score(name, pd.read_csv(f'{DATA_DIR}/{f}')))
    print('done', name)
res = pd.DataFrame(rows)
res.to_csv(f'{DATA_DIR}/roberta_results.csv', index=False)
print(res.round(4).to_string(index=False))

In [ ]:
from google.colab import files
res.to_csv('/content/roberta_results.csv', index=False)
files.download('/content/roberta_results.csv')
print('Done. Send the table above to Claude.')